# Notebook 2: Feature Engineering and Target Construction

This notebook implements the feature engineering stage for the dissertation project on stock market prediction using daily equity data.

The aim is to transform the cleaned dataset from Notebook 1 into a modelling-ready feature set with time-series indicators and three next-day targets. The dissertation pursues a **dual-task design**: a **regression study** predicts SPY's next-day return (continuous), and a **classification study** predicts the direction of that move (binary). Both targets are constructed here from the same feature matrix and consumed by all five models in Notebook 3. The notebook is written as a research workflow: reproducible, chronologically aware, and structured to support later baseline and deep learning experiments.

## Objective

Feature engineering is important in a dissertation setting because the quality and timing of the inputs directly affect the credibility of the downstream modelling results.

In this notebook:
- the cleaned dataset from Notebook 1 is loaded and validated;
- time-series and technical indicators are constructed per symbol;
- three next-day prediction targets are defined — two continuous regression targets and one binary classification target — to support the dissertation's dual-task design;
- invalid warm-up rows are removed cleanly;
- the engineered dataset is saved for model training in Notebook 3, where all three targets are consumed.

## Expected Project Structure

The notebook assumes a project structure similar to:

- `notebooks/`
- `data/processed/`
- `data/features/`
- `results/`

The code below loads the cleaned dataset, engineers predictive features and targets, and saves the resulting feature table for the next stage of the dissertation workflow.


In [1]:
# ============================================================
# Notebook 2: Feature Engineering
# Dissertation Project - Daily Equity Prediction
# ============================================================

# This notebook:
# 1. Loads the cleaned dataset from Notebook 1
# 2. Creates time-series and technical features per symbol
# 3. Creates next-day prediction targets
# 4. Drops invalid rows created by rolling windows / shifts
# 5. Saves the engineered dataset for model training

# Assumes Notebook 1 has already:
# - cleaned the data
# - parsed dates
# - sorted by ["symbol", "date"]
# - saved to PROCESSED_DATA_DIR / "equity_daily_clean.csv"


import pandas as pd
import numpy as np
import sys
from pathlib import Path


def find_project_root(marker="config.py"):
    p = Path.cwd().resolve()
    for _ in range(6):
        if (p / marker).exists():
            return p
        p = p.parent
    raise FileNotFoundError(f"Cannot locate {marker} above {Path.cwd()}")

PROJECT_ROOT = find_project_root()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from config import (
    FEATURE_DATA_DIR, PROCESSED_DATA_DIR,
    MIN_OBSERVATIONS, EXCLUDED_SYMBOLS,
    TRAIN_END_DATE, VAL_END_DATE,
)

# Convert split-date strings from config to Timestamps at point of use.
TRAIN_END_DATE = pd.Timestamp(TRAIN_END_DATE)
VAL_END_DATE   = pd.Timestamp(VAL_END_DATE)

df = pd.read_csv(PROCESSED_DATA_DIR / "equity_daily_clean.csv", parse_dates=["date"])

# Notebook 1 already cleaned and sorted the data. Here we only validate
# the ordering because all rolling and shift features depend on it.
date_diffs = df.groupby("symbol")["date"].diff()
if date_diffs.dropna().lt(pd.Timedelta(0)).any():
    raise ValueError("Input data must be sorted by symbol and date before feature engineering.")

# Features are constructed with information available at the close of day t
# to predict the move on day t+1.
output_path = FEATURE_DATA_DIR / "equity_daily_features.csv"

# TEST_END_DATE is data-driven and cannot be centralised in config.
TEST_END_DATE = df["date"].max()

# Symbols with extremely short histories are not useful for rolling features
# and can add noise to model training.
# Transitional Google share-class tickers around the 2014 split are also
# removed (MIN_OBSERVATIONS and EXCLUDED_SYMBOLS imported from config.py).

symbol_counts = df.groupby("symbol").size()
low_coverage_symbols = symbol_counts[symbol_counts < MIN_OBSERVATIONS].index.tolist()
symbols_to_drop = sorted(set(low_coverage_symbols).union(EXCLUDED_SYMBOLS))

if symbols_to_drop:
    df = df[~df["symbol"].isin(symbols_to_drop)].copy()
    df = df.reset_index(drop=True)

print(f"Project root:  {PROJECT_ROOT}")
print("Dropped symbols before feature engineering:", symbols_to_drop)
print("Remaining symbols:", df["symbol"].nunique())
print(
    f"Split anchors for modelling: train <= {TRAIN_END_DATE.date()}, "
    f"validation <= {VAL_END_DATE.date()}, test <= {TEST_END_DATE.date()}"
)

Project root:  C:\Users\clara\OneDrive\Documents\DissWork
Dropped symbols before feature engineering: ['FOXA', 'GOOAV', 'GOOCV', 'NWSA', 'UW']
Remaining symbols: 16
Split anchors for modelling: train <= 2016-12-30, validation <= 2018-12-31, test <= 2021-03-31


## Technical Indicator Definitions

Three groups of technical indicators require custom implementations to ensure correct per-symbol computation and proper handling of the rolling warm-up period.

**RSI (Relative Strength Index, period 14)** — a bounded momentum oscillator that compares the magnitude of recent gains to recent losses. Wilder's exponential smoothing is used rather than a simple rolling mean, consistent with the original formulation. Values above 70 conventionally indicate overbought conditions; values below 30 indicate oversold conditions.

**MACD (Moving Average Convergence/Divergence)** — a trend-following momentum indicator constructed from the difference between a 12-period and 26-period exponential moving average. The signal line is a 9-period EMA of the MACD itself; the histogram (MACD minus signal) captures the acceleration or deceleration of the prevailing trend.

**Bollinger Bands (period 20, ±2σ)** — volatility envelopes placed two standard deviations above and below a 20-day rolling mean. The band width captures the current volatility regime, while the close's position relative to the upper and lower bands provides a normalised momentum signal that is directly comparable across assets and time periods.

All three indicators are computed using information available at the close of day *t* only, strictly preserving the look-ahead barrier required for valid out-of-sample evaluation.

In [2]:
# ============================================================
# 5. Helper functions
# ============================================================
def compute_rsi(series: pd.Series, window: int = 14) -> pd.Series:
    delta = series.diff()
    gain = delta.clip(lower=0)
    loss = -delta.clip(upper=0)

    # Wilder RSI uses exponentially smoothed average gains and losses.
    avg_gain = gain.ewm(alpha=1 / window, adjust=False, min_periods=window).mean()
    avg_loss = loss.ewm(alpha=1 / window, adjust=False, min_periods=window).mean()

    rs = avg_gain / avg_loss.replace(0, np.nan)
    rsi = 100 - (100 / (1 + rs))
    return rsi


def compute_macd(series: pd.Series, span_short: int = 12, span_long: int = 26, signal_span: int = 9):
    ema_short = series.ewm(span=span_short, adjust=False).mean()
    ema_long = series.ewm(span=span_long, adjust=False).mean()

    macd = ema_short - ema_long
    macd_signal = macd.ewm(span=signal_span, adjust=False).mean()
    macd_hist = macd - macd_signal

    return macd, macd_signal, macd_hist


def compute_bollinger_bands(series: pd.Series, window: int = 20, num_std: int = 2):
    rolling_mean = series.rolling(window=window, min_periods=window).mean()
    rolling_std = series.rolling(window=window, min_periods=window).std()

    upper_band = rolling_mean + num_std * rolling_std
    lower_band = rolling_mean - num_std * rolling_std

    return rolling_mean, upper_band, lower_band

## Feature Construction

Features are computed in-place on the panel, grouped by symbol to prevent any cross-contamination between tickers. Every feature uses only information available at the close of day *t*, strictly preserving the look-ahead barrier required for valid out-of-sample evaluation.

The feature set covers six conceptually distinct groups:

| Group | Features |
|---|---|
| Returns & momentum | 1-day return, log return, lagged returns (1, 2, 3, 5, 10 days) |
| Lagged prices | Close prices lagged 1–10 days |
| Trend | Simple moving averages (5, 10, 20, 50 days) and MA crossover ratios |
| Volatility | Rolling return standard deviation (5, 10, 20 days) |
| Volume | Volume change, moving averages (5, 10, 20 days), volume-to-MA ratios |
| Technical indicators | RSI-14, MACD + signal + histogram, Bollinger Bands (width, position) |

Rows generated during the rolling warm-up period — where windows cannot yet produce valid estimates — will contain `NaN` values and are removed in a subsequent cleaning step.

In [3]:
# ============================================================
# 6. Feature engineering
# ============================================================
# Group once for reuse
g = df.groupby("symbol", group_keys=False)

# -----------------------------
# 6.1 Returns
# -----------------------------
df["return_1d"] = g["close"].pct_change()
df["log_return_1d"] = g["close"].transform(lambda x: np.log(x / x.shift(1)))

# -----------------------------
# 6.2 Lagged returns
# -----------------------------
for lag in [1, 2, 3, 5, 10]:
    df[f"return_lag_{lag}"] = g["return_1d"].shift(lag)

# -----------------------------
# 6.3 Lagged prices
# -----------------------------
for lag in [1, 2, 3, 5, 10]:
    df[f"close_lag_{lag}"] = g["close"].shift(lag)

# -----------------------------
# 6.4 Moving averages
# -----------------------------
for window in [5, 10, 20, 50]:
    df[f"ma_{window}"] = g["close"].transform(lambda x: x.rolling(window, min_periods=window).mean())

# -----------------------------
# 6.5 Rolling volatility
# -----------------------------
for window in [5, 10, 20]:
    df[f"volatility_{window}"] = g["return_1d"].transform(lambda x: x.rolling(window, min_periods=window).std())

# -----------------------------
# 6.6 Rolling return statistics
# -----------------------------
for window in [5, 10, 20]:
    df[f"return_mean_{window}"] = g["return_1d"].transform(lambda x: x.rolling(window, min_periods=window).mean())
    df[f"return_sum_{window}"] = g["return_1d"].transform(lambda x: x.rolling(window, min_periods=window).sum())

# -----------------------------
# 6.7 Price-based ratios / spreads
# -----------------------------
df["intraday_range"] = (df["high"] - df["low"]) / df["open"].replace(0, np.nan)
df["open_close_ratio"] = (df["close"] - df["open"]) / df["open"].replace(0, np.nan)
df["high_close_ratio"] = (df["high"] - df["close"]) / df["close"].replace(0, np.nan)
df["low_close_ratio"] = (df["close"] - df["low"]) / df["close"].replace(0, np.nan)

# -----------------------------
# 6.8 Volume features
# -----------------------------
df["volume_change"] = g["volume"].pct_change()

for window in [5, 10, 20]:
    df[f"volume_ma_{window}"] = g["volume"].transform(lambda x: x.rolling(window, min_periods=window).mean())

df["volume_to_ma5"] = df["volume"] / df["volume_ma_5"]
df["volume_to_ma20"] = df["volume"] / df["volume_ma_20"]

# -----------------------------
# 6.9 RSI
# -----------------------------
df["rsi_14"] = g["close"].transform(lambda x: compute_rsi(x, window=14))

# -----------------------------
# 6.10 MACD
# -----------------------------
macd_parts = g["close"].apply(lambda x: pd.DataFrame(
    dict(zip(["macd", "macd_signal", "macd_hist"], compute_macd(x)))
))

macd_parts = macd_parts.reset_index(level=0, drop=True)
df[["macd", "macd_signal", "macd_hist"]] = macd_parts[["macd", "macd_signal", "macd_hist"]]

# -----------------------------
# 6.11 Bollinger Bands
# -----------------------------
bb_parts = g["close"].apply(lambda x: pd.DataFrame(
    dict(zip(["bb_mid_20", "bb_upper_20", "bb_lower_20"], compute_bollinger_bands(x, window=20)))
))

bb_parts = bb_parts.reset_index(level=0, drop=True)
df[["bb_mid_20", "bb_upper_20", "bb_lower_20"]] = bb_parts[["bb_mid_20", "bb_upper_20", "bb_lower_20"]]

df["bb_width_20"] = (df["bb_upper_20"] - df["bb_lower_20"]) / df["bb_mid_20"]
df["close_to_bb_upper"] = df["close"] / df["bb_upper_20"]
df["close_to_bb_lower"] = df["close"] / df["bb_lower_20"]

# -----------------------------
# 6.12 Moving average crossover features
# -----------------------------
df["ma_5_over_20"] = df["ma_5"] / df["ma_20"]
df["ma_10_over_20"] = df["ma_10"] / df["ma_20"]
df["close_over_ma_5"] = df["close"] / df["ma_5"]
df["close_over_ma_20"] = df["close"] / df["ma_20"]

print("Feature engineering complete.")
print("Current shape:", df.shape)
print("Initial missing values in the first rows are expected from lagged and rolling features before cleaning.")
df.head(10).fillna("warmup")

Feature engineering complete.
Current shape: (69876, 56)
Initial missing values in the first rows are expected from lagged and rolling features before cleaning.


,date,open,high,low,close,volume,symbol,return_1d,log_return_1d,return_lag_1,...,bb_mid_20,bb_upper_20,bb_lower_20,bb_width_20,close_to_bb_upper,close_to_bb_lower,ma_5_over_20,ma_10_over_20,close_over_ma_5,close_over_ma_20
0,2002-05-22,54.50,54.90,54.50,54.89,83600,AAA,warmup,warmup,warmup,...,warmup,warmup,warmup,warmup,warmup,warmup,warmup,warmup,warmup,warmup
1,2002-05-23,54.90,55.10,54.90,54.90,14900,AAA,0.000182,0.000182,warmup,...,warmup,warmup,warmup,warmup,warmup,warmup,warmup,warmup,warmup,warmup
2,2002-05-24,53.69,54.00,53.69,53.81,19100,AAA,-0.019854,-0.020054,0.000182,...,warmup,warmup,warmup,warmup,warmup,warmup,warmup,warmup,warmup,warmup
3,2002-05-28,52.80,52.80,52.10,52.25,5700,AAA,-0.028991,-0.029419,-0.019854,...,warmup,warmup,warmup,warmup,warmup,warmup,warmup,warmup,warmup,warmup
4,2002-05-29,53.00,54.40,53.00,54.20,8600,AAA,0.037321,0.036641,-0.028991,...,warmup,warmup,warmup,warmup,warmup,warmup,warmup,warmup,1.003518,warmup
5,2002-05-30,54.23,54.23,53.78,53.95,8200,AAA,-0.004613,-0.004623,0.037321,...,warmup,warmup,warmup,warmup,warmup,warmup,warmup,warmup,1.002378,warmup
6,2002-05-31,53.37,54.20,53.15,54.05,9600,AAA,0.001854,0.001852,-0.004613,...,warmup,warmup,warmup,warmup,warmup,warmup,warmup,warmup,1.007418,warmup
7,2002-06-03,53.10,53.10,53.10,53.10,1000,AAA,-0.017576,-0.017733,0.001854,...,warmup,warmup,warmup,warmup,warmup,warmup,warmup,warmup,0.992338,warmup
8,2002-06-04,50.90,50.90,50.30,50.50,2400,AAA,-0.048964,-0.050204,-0.017576,...,warmup,warmup,warmup,warmup,warmup,warmup,warmup,warmup,0.949962,warmup
9,2002-06-05,51.40,51.79,51.40,51.50,2900,AAA,0.019802,0.019608,-0.048964,...,warmup,warmup,warmup,warmup,warmup,warmup,warmup,warmup,0.978715,warmup


## Target Variable Construction

Three prediction targets are defined, all based on the next trading day's closing price shifted back by one row within each symbol group to align with day-*t* features. Together they support the dissertation's dual-task design.

| Target | Type | Used for |
|---|---|---|
| `target_next_close` | Continuous (price) | Regression — auxiliary; next-day close is recoverable from `target_next_return` |
| `target_next_return` | Continuous (return) | **Regression study** — primary regression target; stationary and directly comparable across time |
| `target_direction` | Binary (0/1) | **Classification study** — primary classification target |

**Regression task:** `target_next_return` (the percentage change from today's close to tomorrow's close) is the preferred regression target because it is stationary and does not carry the trend component of raw prices, which would make long-horizon comparisons misleading. Tomorrow's closing price is trivially recoverable as `close × (1 + target_next_return)` for applications that require it. `target_next_close` is retained in the dataset as an auxiliary reference.

**Classification task:** `target_direction` is a binary indicator — `1` if tomorrow's close is strictly above today's, `0` otherwise — and is the sole target used for the classification models in Notebook 3.

The final row for each symbol — where no next-day observation exists — will carry `NaN` targets and is removed during the cleaning step below.

In [4]:
# ============================================================
# 7. Target creation
# ============================================================
# Regression target: next-day close
next_close = g["close"].shift(-1)
df["target_next_close"] = next_close

# Regression target: next-day return
df["target_next_return"] = g["return_1d"].shift(-1)

# Classification target: next-day direction
df["target_direction"] = np.where(next_close.notna(), (next_close > df["close"]).astype(int), np.nan)

df[[
    "symbol", "date", "close", "target_next_close", "target_next_return", "target_direction"
]].head(10)

missing_target_rows = df["target_next_close"].isna().sum()
print(
    f"Rows without next-day targets before cleaning: {missing_target_rows} "
    "(expected: last available trading day for each symbol)."
)

Rows without next-day targets before cleaning: 16 (expected: last available trading day for each symbol).


## Removing Warm-Up Rows and Validating the Dataset

Rolling window and lag operations introduce `NaN` values at the start of each symbol's history. The maximum warm-up cost is determined by the longest window in the feature set — the 50-day moving average — together with any additional lag offsets. Target construction adds one further `NaN` row at the end of each symbol's history (the last day has no next-day observation).

All rows containing any `NaN` or infinite value are dropped in a single pass. This conservative approach ensures the modelling dataset is fully complete and avoids any implicit imputation that could introduce bias into downstream results.

In [5]:
# ============================================================
# 8. Clean and validate engineered dataset
# ============================================================
# These invalid rows come from:
# - lagged features
# - rolling windows
# - next-day target shifts

before_drop = df.shape[0]

df = df.replace([np.inf, -np.inf], np.nan)
df = df.dropna().reset_index(drop=True)

after_drop = df.shape[0]

print(f"Rows before cleaning: {before_drop}")
print(f"Rows after cleaning:  {after_drop}")
print(f"Rows removed:         {before_drop - after_drop}")

print("\nFinal shape:", df.shape)
print("\nRemaining missing values:")
print(df.isna().sum().sort_values(ascending=False).head(20))

print("\nSample rows:")
df.head()



Rows before cleaning: 69876
Rows after cleaning:  69050
Rows removed:         826

Final shape: (69050, 59)

Remaining missing values:
date             0
open             0
high             0
low              0
close            0
volume           0
symbol           0
return_1d        0
log_return_1d    0
return_lag_1     0
return_lag_2     0
return_lag_3     0
return_lag_5     0
return_lag_10    0
close_lag_1      0
close_lag_2      0
close_lag_3      0
close_lag_5      0
close_lag_10     0
ma_5             0
dtype: int64

Sample rows:


,date,open,high,low,close,volume,symbol,return_1d,log_return_1d,return_lag_1,...,bb_width_20,close_to_bb_upper,close_to_bb_lower,ma_5_over_20,ma_10_over_20,close_over_ma_5,close_over_ma_20,target_next_close,target_next_return,target_direction
0,2002-08-16,52.25,52.70,52.25,52.70,300,AAA,0.019342,0.019158,0.028856,...,0.176471,0.981665,1.171666,1.057377,1.025369,1.010314,1.068283,51.60,-0.020873,0.0
1,2002-08-21,51.20,51.60,51.20,51.60,400,AAA,-0.020873,-0.021094,0.019342,...,0.180427,0.956813,1.146566,1.046971,1.026412,0.996331,1.043130,51.48,-0.002326,0.0
2,2002-08-22,51.48,51.48,51.48,51.48,100,AAA,-0.002326,-0.002328,-0.020873,...,0.183532,0.951902,1.144259,1.040587,1.030776,0.998720,1.039255,51.00,-0.009324,0.0
3,2002-08-26,50.90,51.00,50.90,51.00,400,AAA,-0.009324,-0.009368,-0.002326,...,0.184703,0.940622,1.132036,1.041513,1.032809,0.986537,1.027490,51.00,0.000000,0.0
4,2002-08-27,51.00,51.00,51.00,51.00,100,AAA,0.000000,0.000000,-0.009324,...,0.179528,0.939261,1.124513,1.034731,1.033186,0.989216,1.023572,49.75,-0.024510,0.0


## Feature Summary

The table below lists all engineered features that will be passed to the modelling stage. Raw OHLCV columns are retained alongside the derived features: tree-based models in Notebook 3 can exploit them directly, and they provide useful context for the sequence-based models introduced in later notebooks.

In [6]:
# ============================================================
# 10. Feature list preview
# ============================================================
target_cols = ["target_next_close", "target_next_return", "target_direction"]
id_cols = ["symbol", "date"]

feature_cols = [col for col in df.columns if col not in id_cols + target_cols]

print("Number of feature columns:", len(feature_cols))
print(feature_cols)

Number of feature columns: 54
['open', 'high', 'low', 'close', 'volume', 'return_1d', 'log_return_1d', 'return_lag_1', 'return_lag_2', 'return_lag_3', 'return_lag_5', 'return_lag_10', 'close_lag_1', 'close_lag_2', 'close_lag_3', 'close_lag_5', 'close_lag_10', 'ma_5', 'ma_10', 'ma_20', 'ma_50', 'volatility_5', 'volatility_10', 'volatility_20', 'return_mean_5', 'return_sum_5', 'return_mean_10', 'return_sum_10', 'return_mean_20', 'return_sum_20', 'intraday_range', 'open_close_ratio', 'high_close_ratio', 'low_close_ratio', 'volume_change', 'volume_ma_5', 'volume_ma_10', 'volume_ma_20', 'volume_to_ma5', 'volume_to_ma20', 'rsi_14', 'macd', 'macd_signal', 'macd_hist', 'bb_mid_20', 'bb_upper_20', 'bb_lower_20', 'bb_width_20', 'close_to_bb_upper', 'close_to_bb_lower', 'ma_5_over_20', 'ma_10_over_20', 'close_over_ma_5', 'close_over_ma_20']


## Saving the Engineered Dataset

The completed feature table is written to `data/features/equity_daily_features.csv`. This file is the primary input for all modelling notebooks and should not be regenerated unless the feature set or underlying data changes.

In [7]:
# ============================================================
# 11. Save engineered dataset and output
# ============================================================
df.to_csv(output_path, index=False)

print(f"Saved engineered dataset to: {output_path}")

print("Notebook 2 complete.")
print(f"Engineered dataset shape: {df.shape}")
print(f"Feature count: {len(feature_cols)}")
df.head()

Saved engineered dataset to: C:\Users\clara\OneDrive\Documents\DissWork\data\features\equity_daily_features.csv
Notebook 2 complete.
Engineered dataset shape: (69050, 59)
Feature count: 54


,date,open,high,low,close,volume,symbol,return_1d,log_return_1d,return_lag_1,...,bb_width_20,close_to_bb_upper,close_to_bb_lower,ma_5_over_20,ma_10_over_20,close_over_ma_5,close_over_ma_20,target_next_close,target_next_return,target_direction
0,2002-08-16,52.25,52.70,52.25,52.70,300,AAA,0.019342,0.019158,0.028856,...,0.176471,0.981665,1.171666,1.057377,1.025369,1.010314,1.068283,51.60,-0.020873,0.0
1,2002-08-21,51.20,51.60,51.20,51.60,400,AAA,-0.020873,-0.021094,0.019342,...,0.180427,0.956813,1.146566,1.046971,1.026412,0.996331,1.043130,51.48,-0.002326,0.0
2,2002-08-22,51.48,51.48,51.48,51.48,100,AAA,-0.002326,-0.002328,-0.020873,...,0.183532,0.951902,1.144259,1.040587,1.030776,0.998720,1.039255,51.00,-0.009324,0.0
3,2002-08-26,50.90,51.00,50.90,51.00,400,AAA,-0.009324,-0.009368,-0.002326,...,0.184703,0.940622,1.132036,1.041513,1.032809,0.986537,1.027490,51.00,0.000000,0.0
4,2002-08-27,51.00,51.00,51.00,51.00,100,AAA,0.000000,0.000000,-0.009324,...,0.179528,0.939261,1.124513,1.034731,1.033186,0.989216,1.023572,49.75,-0.024510,0.0


## Conclusion

This notebook has produced a clean, modelling-ready feature dataset from the raw daily equity panel. The key outcomes are:

- **54 engineered features** spanning returns, momentum, trend, volatility, volume, and technical indicators.
- **16 symbols** retained after filtering on minimum observation count.
- **69,050 rows** covering 1998–2021, with warm-up and missing-target rows cleanly removed.
- Three prediction targets constructed from the same feature matrix:
  - `target_next_return` — next-day percentage return; **primary regression target** in Notebook 3.
  - `target_next_close` — raw next-day closing price; auxiliary regression reference.
  - `target_direction` — binary up/down indicator; **primary classification target** in Notebook 3.
- Strict chronological ordering preserved throughout — no future information has been used in any feature or target calculation.

The dual-target design means Notebook 3 can train and evaluate all five model families on **both tasks** using the same features, the same preprocessing pipeline, and the same chronological train/validation/test splits — ensuring a fair, directly comparable experimental design across regression and classification.

The engineered dataset is saved to `data/features/equity_daily_features.csv` and is ready for model training in Notebook 3.